# 🛣️ Pengujian & Perbandingan Model FastSAM-s vs FastSAM-x
## Segmentasi Kerusakan Jalan dan Fasilitas Kawasan

Notebook ini membandingkan dan menguji secara terpisah kinerja dua varian Segment Anything Model yang tersedia di `playground/models/`:
1. **FastSAM-s.pt** (~23.8 MB) - Varian Small (ringan & berkecepatan tinggi)
2. **FastSAM-x.pt** (~144.9 MB) - Varian Extra Large (kontur tepi lebih detail)

### 🎯 Target Objek & Kerusakan:
- **`pothole`**: Lubang jalan dan genangan air pada lubang aspal.
- **`damaged_road`**: Koridor permukaan aspal jalan yang rusak/hancur.
- **`broken_convex_mirror`**: Kaca cembung jalan raya yang pecah/rusak bagian tengahnya.

### 📊 Output yang Dihasilkan:
- Visualisasi terpisah `*_fastsam_s_segmented.jpg` dan `*_fastsam_x_segmented.jpg`
- Panel komparasi side-by-side `*_comparison_s_vs_x.jpg` `[Original | FastSAM-s | FastSAM-x]`
- File metadata JSON terpisah `playground/image-output/detections_metadata.json`

In [ ]:
# 1. Setup Dependensi & Path
import os
import time
import json
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import FastSAM

BASE_DIR = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
IMAGES_DIR = BASE_DIR / "images"
MODELS_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "image-output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[STATUS] Device Komputasi : {device}")
print(f"[STATUS] Folder Gambar   : {IMAGES_DIR}")
print(f"[STATUS] Folder Model    : {MODELS_DIR}")
print(f"[STATUS] Folder Output   : {OUTPUT_DIR}")

In [ ]:
# 2. Inisialisasi Model FastSAM-s dan FastSAM-x
model_s_path = MODELS_DIR / "FastSAM-s.pt"
model_x_path = MODELS_DIR / "FastSAM-x.pt"

print(f"Memuat FastSAM-s ({model_s_path.stat().st_size / (1024*1024):.1f} MB)...")
model_s = FastSAM(str(model_s_path))

print(f"Memuat FastSAM-x ({model_x_path.stat().st_size / (1024*1024):.1f} MB)...")
model_x = FastSAM(str(model_x_path))

print("✓ FastSAM-s dan FastSAM-x siap digunakan!")

## 🔍 Eksperimen 1: Perbandingan pada `jalan-hancur.png` (Jalan Rusak & Pothole Air)

In [ ]:
# Memuat hasil visualisasi terpisah dan komparasi side-by-side
comp_1 = cv2.imread(str(OUTPUT_DIR / "jalan-hancur_comparison_s_vs_x.jpg"))

plt.figure(figsize=(18, 6))
plt.imshow(cv2.cvtColor(comp_1, cv2.COLOR_BGR2RGB))
plt.title("Komparasi: Original vs FastSAM-s vs FastSAM-x pada jalan-hancur.png", fontsize=14)
plt.axis('off')
plt.show()

## 🔍 Eksperimen 2: Perbandingan pada `kaca-cembung-pecah.jpg` (Cermin Cembung Pecah)

In [ ]:
comp_2 = cv2.imread(str(OUTPUT_DIR / "kaca-cembung-pecah_comparison_s_vs_x.jpg"))

plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(comp_2, cv2.COLOR_BGR2RGB))
plt.title("Komparasi: Original vs FastSAM-s vs FastSAM-x pada kaca-cembung-pecah.jpg", fontsize=14)
plt.axis('off')
plt.show()

## 🔍 Eksperimen 3: Perbandingan pada `pothole-banyak.png` (Tampilan Udara Multi-Pothole)

In [ ]:
comp_3 = cv2.imread(str(OUTPUT_DIR / "pothole-banyak_comparison_s_vs_x.jpg"))

plt.figure(figsize=(18, 5))
plt.imshow(cv2.cvtColor(comp_3, cv2.COLOR_BGR2RGB))
plt.title("Komparasi: Original vs FastSAM-s vs FastSAM-x pada pothole-banyak.png", fontsize=14)
plt.axis('off')
plt.show()

## 🔍 Eksperimen 4: Perbandingan pada `pothole-kecil.jpg` (Pothole Dekat Trotoar)

In [ ]:
comp_4 = cv2.imread(str(OUTPUT_DIR / "pothole-kecil_comparison_s_vs_x.jpg"))

plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(comp_4, cv2.COLOR_BGR2RGB))
plt.title("Komparasi: Original vs FastSAM-s vs FastSAM-x pada pothole-kecil.jpg", fontsize=14)
plt.axis('off')
plt.show()

## 📊 Analisis Metadata JSON Terpisah (`FastSAM-s` vs `FastSAM-x`)

In [ ]:
metadata_file = OUTPUT_DIR / "detections_metadata.json"
with open(metadata_file) as f:
    data = json.load(f)

print(f"File Metadata: {metadata_file}")
print(f"Generated at : {data.get('generated_at')}\n")

for r in data.get('results', []):
    print(f"=======================================================================")
    print(f"📷 GAMBAR: {r['image_name']} ({r['image_size']['width']}x{r['image_size']['height']} px)")
    print(f"=======================================================================")
    
    # Data FastSAM-s
    s = r['fastsam_s']
    print(f"  ⚡ Model [FastSAM-s] ({s['model_size_mb']} MB):")
    print(f"     - Kecepatan Inferensi: {s['inference_time_ms']} ms")
    print(f"     - Raw Kandidat Masks: {s['candidates_count']}")
    print(f"     - Area Terdeteksi    : {s['detections_count']}")
    print(f"     - File Output        : {s['visual_file']}")
    
    # Data FastSAM-x
    x = r['fastsam_x']
    print(f"  🎯 Model [FastSAM-x] ({x['model_size_mb']} MB):")
    print(f"     - Kecepatan Inferensi: {x['inference_time_ms']} ms")
    print(f"     - Raw Kandidat Masks: {x['candidates_count']}")
    print(f"     - Area Terdeteksi    : {x['detections_count']}")
    print(f"     - File Output        : {x['visual_file']}")
    
    print(f"  ⚖️ Perbandingan Kecepatan (Speedup FastSAM-s): {r['comparison']['speedup_factor']}x lebih cepat")
    print(f"  🖼️ File Side-by-Side: {r['comparison']['comparison_image']}\n")

## 📝 Kesimpulan Evaluasi Komparatif FastSAM-s vs FastSAM-x

| Parameter | FastSAM-s | FastSAM-x | Rekomendasi Penggunaan |
|---|---|---|---|
| **Ukuran Model** | ~23.8 MB | ~144.9 MB | FastSAM-s hemat storage & memori |
| **Kecepatan Inferensi** | 3-5x lebih cepat | Standar | FastSAM-s ideal untuk Video CCTV / Drone Real-Time |
| **Presisi Mask Tepi** | Baik | Sangat Halus & Detail | FastSAM-x ideal untuk Audit Statis / Estimasi Luas Aspal |
| **Kandidat Segmentasi** | Cenderung lebih banyak proposal awal | Proposal lebih terfokus | Dua model sama-sama membutuhkan Prompt (Box/Point) |